# Supplementary Figure S3 — model response fields

Run cells in order. Calculated panels are saved under `figures/`.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "notebooks").is_dir() and (p / "data").is_dir())
sys.path.insert(0, str(ROOT / "src"))
FIGURES = ROOT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
SAVED_RESULTS = ROOT / "saved_results"

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
plt.style.use(ROOT / "notebooks/ncb.mplstyle")
plt.close("all")
np.random.seed(0)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import plotting as p
from stim_gen import makeCHIRPstim, makePPFstim

import jax.numpy as jnp

def nmax(x):
    return x / np.nanmax(x)

## Compute or load preference fields

In [ ]:
RUN_EXPENSIVE = False  # Set True to simulate both models and refresh the fields.
CACHE = (ROOT / "results" if RUN_EXPENSIVE else SAVED_RESULTS) / "figS3_fields.npz"

if RUN_EXPENSIVE:
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    from model.model_20210125_params import Params
    from model.model_20210125_structure import run_model
    from model.model_20210125_params_fix import Params as Params_fix
    from model.model_20210125_structure_fix import run_model as run_model_fix
    stim_matrix, stim_dict, ppau, pdur = makePPFstim(ppauMax=80, pdurMax=80, ppauMin=0, pdurMin=0, cdur=200, cpau=200, Fs=1000, step=1, ppau=None, pdur=None)
    model_output = run_model(stim_matrix, Params(), verbose_output=True, return_dict=True)
    model_output_fix = run_model_fix(stim_matrix, Params_fix(), verbose_output=True, return_dict=True)

    ppf = np.nanmean(model_output['ln4'], axis=0).reshape((80,80))
    ppf_fix = np.nanmean(model_output_fix['ln4'], axis=0).reshape((80,80))

    pdur = np.unique(stim_dict['pdur'])
    ppau = np.unique(stim_dict['ppau'])

    stim_matrix, stim_dict, cpau, cdur = makeCHIRPstim(
        ppau=20,
        pdur=20,
        Fs=1000,
        num_steps=64,
        stim_len=5_000,
    )

    model_output = run_model(stim_matrix, Params(), verbose_output=True, return_dict=True)
    model_output_fix = run_model_fix(stim_matrix, Params_fix(), verbose_output=True, return_dict=True)

    cpf = np.nanmean(model_output['ln4'], axis=0).reshape((64,64))
    cpf_fix = np.nanmean(model_output_fix['ln4'], axis=0).reshape((64,64))

    cdur = np.unique(stim_dict['cdur']) / 1000
    cpau = np.unique(stim_dict['cpau']) / 1000
    np.savez_compressed(CACHE, ppf=ppf, ppf_fix=ppf_fix, cpf=cpf, cpf_fix=cpf_fix,
                        pdur=pdur, ppau=ppau, cdur=cdur, cpau=cpau)
else:
    if not CACHE.exists():
        raise FileNotFoundError(f"{CACHE} is missing. Set RUN_EXPENSIVE = True and run this cell once.")
    with np.load(CACHE) as cached:
        ppf, ppf_fix = cached["ppf"], cached["ppf_fix"]
        cpf, cpf_fix = cached["cpf"], cached["cpf_fix"]
        pdur, ppau = cached["pdur"], cached["ppau"]
        cdur, cpau = cached["cdur"], cached["cpau"]

## Plot normalized fields and signed differences

In [ ]:
plt.figure(figsize=(12, 8))
plt.subplot(231)
plt.pcolormesh(pdur, ppau, p.nmax(ppf), cmap='RdBu', rasterized=True, shading='gouraud')
plt.axis('square')
plt.xlim(0, 80)
plt.ylim(0, 80)
plt.clim(-1, 1)
plt.xticks([0, 40, 80])
plt.yticks([0, 40, 80])
plt.ylabel('Pulse pause [ms]')
plt.title('Original model')

plt.subplot(232)
plt.pcolormesh(pdur, ppau, p.nmax(ppf_fix), cmap='RdBu_r', rasterized=True, shading='gouraud')
plt.axis('square')
plt.xlim(0, 80)
plt.ylim(0, 80)
plt.clim(-1, 1)
plt.xticks([0, 40, 80])
plt.yticks([0, 40, 80], labels=[])
plt.title('Modified model')
plt.xlabel('Pulse duration [ms]')

plt.subplot(233)
pcm = plt.pcolormesh(pdur, ppau, p.nmax(ppf) - p.nmax(ppf_fix), cmap='RdBu', rasterized=True, shading='gouraud')
plt.axis('square')
plt.xlim(0, 80)
plt.ylim(0, 80)
plt.clim(-1, 1)
plt.xticks([0, 40, 80])
plt.yticks([0, 40, 80], labels=[])
plt.title('Difference')
cax = plt.gca().inset_axes([0.85, 0.07, 0.04, 0.3])
cb = plt.gcf().colorbar(pcm, cax=cax, orientation='vertical')
cb.set_ticklabels(['Modified', 'Identical', 'Original'])

plt.subplot(234)
plt.pcolormesh(cdur, cpau, p.nmax(cpf), cmap='RdBu', rasterized=True, shading='gouraud')
plt.axis('square')
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.clim(-1, 1)
plt.xticks([0, 0.5, 1])
plt.yticks([0, 0.5, 1])
plt.ylabel('Chirp pause [s]')

plt.subplot(235)
plt.pcolormesh(cdur, cpau, p.nmax(cpf_fix), cmap='RdBu_r', rasterized=True, shading='gouraud')
plt.axis('square')
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.clim(-1, 1)
plt.xticks([0, 0.5, 1])
plt.yticks([0, 0.5, 1], labels=[])
plt.xlabel('Chirp duration [s]')

plt.subplot(236)
pcm = plt.pcolormesh(cdur, cpau, p.nmax(cpf) - p.nmax(cpf_fix), cmap='RdBu', rasterized=True, shading='gouraud')
plt.axis('square')
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.clim(-1, 1)
plt.xticks([0, 0.5, 1])
plt.yticks([0, 0.5, 1], labels=[])
cax = plt.gca().inset_axes([0.85, 0.07, 0.04, 0.3])
cb = plt.gcf().colorbar(pcm, cax=cax, orientation='vertical')
cb.set_ticklabels(['Modified', 'Identical', 'Original'])

plt.tight_layout()
plt.savefig(f'{FIGURES}/fig_S3.pdf', bbox_inches='tight', facecolor=None)